# Create helper methods to:
 1. Load the CoDocGen model to generate documentation for a function
 2. Method that uses the CoDocModel model to generate documentation of the given code (as string)
 3. Load the the-stack dataset and extract only the C++ and python code from it.
 4. load the github/tree-sitter to create abstract syntx trees of given code and lanugage
 5. Create ASTs for the given code.

# Install all the necessary packages

In [ ]:
pip install transformers torch accelerate  datasets  tree-sitter tree-sitter-languages

In [ ]:
from google.colab import userdata
import os

# Try to retrieve the Hugging Face API key from Colab's secrets manager
HUGGING_FACE_KEY = userdata.get('HUGGING_FACE_KEY')

if HUGGING_FACE_KEY is None:
    print("WARNING: Hugging Face API key (HUGGING_FACE_KEY) not found in Colab secrets.")

    # Prompt user for input if not found in secrets
    HUGGING_FACE_KEY = input("Please enter your Hugging Face API Key: ")
    if not HUGGING_FACE_KEY:
        print("Hugging Face API Key was not provided. Some models might not load correctly.")
        import sys
        sys.exit()
    else:
        os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
        print("Hugging Face API key received from input.")
else:
    # Set the environment variable for Hugging Face if found in secrets
    os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
    print("Hugging Face API key loaded successfully from secrets.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

def load_codocgen_model():
    """Loads the CoDocGen model and tokenizer."""
    model_name = "CoDoCGen/CoDoCGen-7B"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
    return tokenizer, model

tokenizer, model = load_codocgen_model()
print("CoDocGen model and tokenizer loaded.")

In [ ]:
def generate_documentation(code: str, max_length=512, num_return_sequences=1) -> list:
    """
    Generates documentation for a given code snippet using the CoDoCGen model.

    Args:
        code (str): The code for which to generate documentation.
        max_length (int): The maximum length of the generated documentation.
        num_return_sequences (int): The number of different documentation sequences to generate.

    Returns:
        list: A list of generated documentation strings.
    """
    input_text = f"document code: {code_snippet}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

    # Generate documentation
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_return_sequences=num_return_sequences,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    generated_docs = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_docs

In [ ]:
# Test the documentation generation
sample_code = """
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
"""

print("Generating documentation for the following code:")
print(sample_code)

documentation = generate_documentation(sample_code)

print("\nGenerated Documentation:")
for doc in documentation:
    print(doc)

## Load and Filter Code Dataset

A function to load a `codeparrot/github-code` dataset and filter it to include only C++ and Python code. The `language` column has the type of laguage

In [ ]:
from datasets import load_dataset

def load_and_filter_code_dataset(dataset_name="codeparrot/github-code", languages=None):
    """
    Loads a code dataset and filters it by specified languages.

    Args:
        dataset_name (str): The name of the dataset to load (e.g., "codeparrot/github-code").
        languages (list): A list of programming languages to filter by (e.g., ['C++', 'Python']).
                          If None, no language filtering is applied.

    Returns:
        datasets.Dataset: The filtered dataset.
    """
    if languages is None:
        languages = []

    print(f"Loading dataset: {dataset_name}")
    dataset = load_dataset(dataset_name, split="train") # Assuming 'train' split for demonstration

    if languages:
        print(f"Filtering dataset for languages: {', '.join(languages)}")
        # Filter by the 'language' column, assuming it exists
        filtered_dataset = dataset.filter(lambda example: example['language'] in languages)
        return filtered_dataset
    else:
        return dataset

### Test: Loading and Filtering Code

Use the `load_and_filter_code_dataset` function to get only C++ and Python code from the `codeparrot/github-code` dataset.

In [ ]:
try:
    cpp_python_dataset = load_and_filter_code_dataset(languages=['C++', 'Python'])

    print(f"\nTotal examples in filtered dataset: {len(cpp_python_dataset)}")
    print("\nFirst 5 examples (showing language and a snippet of code):")
    for i in range(min(5, len(cpp_python_dataset))):
        example = cpp_python_dataset[i]
        print(f"---\nLanguage: {example['language']}\nCode Snippet: {example['code'][:200]}...")
except RuntimeError as e:
    if "Dataset scripts are no longer supported" in str(e):
        print(f"\nError loading dataset: {e}")
        print("\nIt appears the `codeparrot/github-code` dataset cannot be loaded directly via script anymore.")
        print("To fix this, please modify the `load_and_filter_code_dataset` function in cell `21490850`.")
        print("You might need to specify a `config_name` (e.g., 'all' or 'code_x_m') and potentially use `streaming=True` if the dataset is very large, like so:")
        print("    `dataset = load_dataset(dataset_name, 'all', split=\"train\", streaming=True)`")
        print("Alternatively, you might need to find a different version of the dataset or a more compatible dataset for demonstration purposes.")
    else:
        raise e

### Generate Abstract Syntax Tree (AST) with Tree-sitter

To generate an Abstract Syntax Tree (AST) for code, we can use the `tree-sitter` library. This involves installing the library and specific language parsers.

In [ ]:
import os
from tree_sitter import Language, Parser

# Define the directory to store grammars
GRAMMAR_DIR = "tree-sitter-grammars"
BUILD_DIR = os.path.join(GRAMMAR_DIR, "build")

if not os.path.exists(GRAMMAR_DIR):
    os.makedirs(GRAMMAR_DIR)
if not os.path.exists(BUILD_DIR):
    os.makedirs(BUILD_DIR)

# Clone grammars if they don't exist
if not os.path.exists(os.path.join(GRAMMAR_DIR, "python")):
    print("Cloning tree-sitter-python grammar...")
    !git clone https://github.com/tree-sitter/tree-sitter-python {GRAMMAR_DIR}/python
if not os.path.exists(os.path.join(GRAMMAR_DIR, "cpp")):
    print("Cloning tree-sitter-cpp grammar...")
    !git clone https://github.com/tree-sitter/tree-sitter-cpp {GRAMMAR_DIR}/cpp

# Path to the compiled shared library
LIBRARY_PATH = os.path.join(BUILD_DIR, "my-languages.so")

# Build the shared library (if it doesn't exist or grammars have changed)
if not os.path.exists(LIBRARY_PATH):
    print("Building tree-sitter language library...")
    Language.build_library(
        LIBRARY_PATH,
        [
            os.path.join(GRAMMAR_DIR, "python"),
            os.path.join(GRAMMAR_DIR, "cpp"),
        ]
    )
else:
    print("Tree-sitter language library already built.")

# Load the compiled languages
PYTHON_LANGUAGE = Language(LIBRARY_PATH, "python")
CPP_LANGUAGE = Language(LIBRARY_PATH, "cpp")

print("Tree-sitter Python and C++ languages loaded successfully.")

### Function generate the AST Given the code
The generate_ast_with_tree_sitter function will use the manually loaded Language objects and generate AST for the given code

In [ ]:
from tree_sitter import Language, Parser
import tree_sitter_languages as ts_languages

def generate_ast_with_tree_sitter(code_snippet: str, language_name: str):
    """
    Generates an Abstract Syntax Tree (AST) for a given code snippet
    using tree-sitter for the specified language.

    Args:
        code_snippet (str): The code for which to generate the AST.
        language_name (str): The name of the programming language (e.g., 'python', 'cpp').

    Returns:
        tree_sitter.Tree: The generated AST.
    """
    try:
        # Load the language dynamically using tree-sitter-languages.
        # This function directly returns a tree_sitter.Language object.
        language = ts_languages.get_language(language_name)

        parser = Parser()
        parser.set_language(language)

        tree = parser.parse(bytes(code_snippet, "utf8"))
        return tree
    except Exception as e:
        print(f"Error generating AST for {language_name} code: {e}")
        print(f"Please ensure the parser for '{language_name}' is installed and available.")
        return None

def print_ast_node(node, indent=0):
    """
    Helper function to print AST nodes recursively.
    """
    print(f"{'  ' * indent}{node.type} [start={node.start_point}, end={node.end_point}]")
    for child in node.children:
        print_ast_node(child, indent + 1)

### Test: Generating ASTs

Let's demonstrate `generate_ast_with_tree_sitter` with a Python function and a C++ snippet.

In [ ]:
# Example 1: Python Code
python_code = """
def greet(name):
    print(f"Hello, {name}!")
"""

print("Generating AST for Python code:")
python_ast = generate_ast_with_tree_sitter(python_code, 'python')
if python_ast:
    print_ast_node(python_ast.root_node)

print("\n" + "="*50 + "\n")

# Example 2: C++ Code
cpp_code = """
#include <iostream>

int main() {
    std::cout << "Hello from C++!" << std::endl;
    return 0;
}
"""

print("Generating AST for C++ code:")
cpp_ast = generate_ast_with_tree_sitter(cpp_code, 'cpp')
if cpp_ast:
    print_ast_node(cpp_ast.root_node)